# Module 5 · Evaluation & Safety (panel companion, run-only)

Run against a **known-good** reference agent so the demo never depends on anyone's unfinished
edits. We run a small red-team probe set and walk the evaluation checklist
(`templates/eval_safety_checklist.md`).

In [ ]:
# Make the top-level ha_workshop package importable and anchor relative paths to the repo root.
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'healthagent').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print('repo root:', ROOT)

In [ ]:
from healthagent.llm.client import get_client
from healthagent.safety.probes import run_all
from ha_workshop.solutions import agent_config as known_good   # reference agent, isolated from student edits
client = get_client('scripted')   # deterministic teaching backend — reproducible eval rows
rows = run_all(lambda q: known_good.run(q, client=client))
for r in rows:
    print(f"{r['id']:7s} {r['category']:9s} expect={r['expectation']:8s} "
          f"refused={r['refused']!s:5s} grounded={r['grounded']!s:5s} -> {'PASS' if r['passed'] else 'FAIL'}")

### Ground truth vs. evidence
The generator *seeded* night screen-time as the driver of `u01`'s poor sleep (synthetic ground
truth, see `data/DATA_CARD.md`). The agent never sees that rule — it only observes an
**association**, plus a seeded confound (a `deadline` event and `caffeine_after_6pm`). That gap
is exactly why answers say *"most plausible contributor in this synthetic dataset"* and carry a
correlation-≠-causation caveat.

In [ ]:
# Optional: test YOUR OWN agent from Module 4 (may differ if your TODOs aren't complete).
# from ha_workshop import reload_ha_workshop
# _, cfg = reload_ha_workshop()
# for r in run_all(lambda q: cfg.run(q, client=client)): print(r['id'], 'PASS' if r['passed'] else 'FAIL')

In [ ]:
# Optional — re-run the SAME red-team probe set on a LIVE backend (set HA_TRY_LIVE=1).
import os
if os.getenv('HA_TRY_LIVE') == '1':
    live = get_client('auto')
    print('live tier:', live.provider)
    for r in run_all(lambda q: known_good.run(q, client=live)):
        print(f"{r['id']:7s} {'PASS' if r['passed'] else 'FAIL'}")
else:
    print("Skipping live backend cell (set HA_TRY_LIVE=1 to try a live backend).")